# 🏨 StaySmart Hotels — Feature Engineering Capstone
**Predicting Booking Cancellation Risk**

- **Target**: `is_canceled` (binary classification)
- **Dataset**: Hotel Bookings (public Kaggle version)
- **Goal**: Prove that feature engineering + preprocessing drives performance

In [ ]:
# Install any missing packages
!pip install -q category_encoders scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, PowerTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.datasets import make_classification
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import KBinsDiscretizer

import category_encoders as ce

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
RANDOM_STATE = 42
print('✅ Imports complete')

In [ ]:
# ── Download Dataset ──────────────────────────────────────────────────────────
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
df_raw = pd.read_csv(url)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print('\nTarget distribution:')
print(df_raw['is_canceled'].value_counts(normalize=True))

---
## Task 1: Baseline Model + "What is a Feature?"

In [ ]:
# ── What is a Feature? ────────────────────────────────────────────────────────
print("""
WHAT IS A FEATURE?
==================
A feature (also called a predictor or input variable) is any measurable property
of an observation that is used as input to a machine learning model. Features
encode information the model uses to learn patterns and make predictions.

GOOD FEATURE — lead_time:
  The number of days between booking and arrival. Guests who book far in advance
  are more likely to cancel as plans change. This has strong predictive signal
  for cancellations and is directly measurable and interpretable.

BAD FEATURE — reservation_status_date:
  The date the reservation status was last set. This is a consequence of the
  cancellation (target), not a cause — using it would cause data leakage and
  artificially inflate model performance. It provides no useful predictive value
  in a real deployment where the future is unknown.
""")

In [ ]:
# ── Baseline Model ────────────────────────────────────────────────────────────
TARGET = 'is_canceled'

# Drop leaky columns
LEAKY_COLS = ['reservation_status', 'reservation_status_date']
df = df_raw.drop(columns=LEAKY_COLS).copy()

# Simple feature set for baseline
NUM_COLS_BASE = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
                 'adults', 'children', 'babies', 'previous_cancellations',
                 'booking_changes', 'adr', 'total_of_special_requests']
CAT_COLS_BASE = ['hotel', 'meal', 'market_segment', 'distribution_channel',
                 'deposit_type', 'customer_type']

X = df[NUM_COLS_BASE + CAT_COLS_BASE].copy()
y = df[TARGET]

# Handle categoricals simply
for c in CAT_COLS_BASE:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))

X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline_model = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

baseline_acc  = accuracy_score(y_test, y_pred)
baseline_auc  = roc_auc_score(y_test, y_prob)
baseline_f1   = f1_score(y_test, y_pred)

print('=== BASELINE RESULTS ===')
print(f'Accuracy : {baseline_acc:.4f}')
print(f'ROC-AUC  : {baseline_auc:.4f}')
print(f'F1 Score : {baseline_f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Canceled', 'Canceled']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Not Canceled', 'Canceled']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Baseline Confusion Matrix')
plt.tight_layout()
plt.savefig('baseline_confusion_matrix.png', dpi=150)
plt.show()

---
## Task 2: Curse of Dimensionality Demo

In [ ]:
from sklearn.datasets import make_classification
from scipy.spatial.distance import cdist

dims = [2, 10, 50, 200]
n_samples = 500

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ratio_means = []
for d in dims:
    X_syn, _ = make_classification(n_samples=n_samples, n_features=d,
                                    n_informative=max(2, d//2),
                                    n_redundant=0, random_state=RANDOM_STATE)
    # pairwise distances on 200 random pairs
    idx = np.random.choice(n_samples, size=200, replace=False)
    dists = cdist(X_syn[idx[:100]], X_syn[idx[100:]], metric='euclidean').flatten()
    axes[0].hist(dists, bins=30, alpha=0.5, label=f'd={d}')
    # min/max ratio
    all_dists = cdist(X_syn[:100], X_syn[:100], metric='euclidean')
    np.fill_diagonal(all_dists, np.inf)
    min_d = all_dists.min(axis=1)
    np.fill_diagonal(all_dists, 0)
    max_d = all_dists.max(axis=1)
    ratio_means.append(np.mean(min_d / max_d))

axes[0].set_xlabel('Euclidean Distance')
axes[0].set_ylabel('Count')
axes[0].set_title('Distance Distribution by Dimensionality')
axes[0].legend()

axes[1].plot(dims, ratio_means, marker='o', color='crimson', linewidth=2)
axes[1].set_xlabel('Number of Dimensions')
axes[1].set_ylabel('Mean (min_dist / max_dist)')
axes[1].set_title('Nearest/Farthest Neighbor Ratio (→1 = indistinguishable)')
for d, r in zip(dims, ratio_means):
    axes[1].annotate(f'{r:.2f}', (d, r), textcoords='offset points', xytext=(5, 5))

plt.tight_layout()
plt.savefig('curse_of_dimensionality.png', dpi=150)
plt.show()

print("""
OBSERVATIONS — Curse of Dimensionality
=======================================
1. DISTANCE SPREAD COLLAPSES: As dimensions increase from 2→200, the histogram
   of pairwise distances becomes narrower and shifts right. All points start to
   look equally far apart — a phenomenon called 'distance concentration'.

2. MIN/MAX RATIO INCREASES TOWARD 1: The ratio of nearest-to-farthest neighbor
   distance approaches 1 in high dimensions. This means nearest neighbors become
   meaningless — every point is about as far from every other point.

3. LEARNING BECOMES HARDER: Distance-based algorithms (KNN, KMeans, SVMs with
   RBF kernels) rely on proximity as a proxy for similarity. When distances
   lose meaning, these models degrade severely.

4. DATA SPARSITY EXPLODES: In 2D, 100 samples cover a grid reasonably. In 200D,
   you'd need exponentially more samples to maintain coverage — infeasible in
   practice.

5. FEATURE ENGINEERING NECESSITY: By carefully selecting, constructing, and
   reducing features, we keep only informative dimensions — reducing sparsity,
   preserving distance meaning, and making models more stable and interpretable.
""")

---
## Task 3: Numeric Preprocessing

In [ ]:
NUM_COLS = ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights',
            'total_of_special_requests', 'previous_cancellations',
            'adults', 'booking_changes']

df_num = df[NUM_COLS].fillna(df[NUM_COLS].median()).copy()

# ── 1. Binning ─────────────────────────────────────────────────────────────
# Equal-width bins for lead_time
df_num['lead_time_bin_ew'] = pd.cut(df_num['lead_time'], bins=5,
                                     labels=['VeryShort','Short','Medium','Long','VeryLong'])
# Quantile bins for adr
df_num['adr_bin_q'] = pd.qcut(df_num['adr'], q=4,
                               labels=['Budget','Economy','Standard','Premium'],
                               duplicates='drop')
print('Lead-time bin counts:\n', df_num['lead_time_bin_ew'].value_counts())
print('\nADR quantile bin counts:\n', df_num['adr_bin_q'].value_counts())

# ── 2. Binarization ────────────────────────────────────────────────────────
adr_threshold = df_num['adr'].quantile(0.75)
df_num['high_value_customer'] = (df_num['adr'] > adr_threshold).astype(int)
print(f'\nHigh-value customer threshold (75th pct): {adr_threshold:.2f}')
print(df_num['high_value_customer'].value_counts())

In [ ]:
# ── 3. Scaling Comparison ─────────────────────────────────────────────────
SCALE_COLS = ['lead_time', 'adr', 'stays_in_week_nights', 'previous_cancellations']
df_scale = df_num[SCALE_COLS].copy()

scalers = {
    'MinMaxScaler': MinMaxScaler(),
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler()
}

scaled_dfs = {}
for name, scaler in scalers.items():
    scaled_dfs[name] = pd.DataFrame(
        scaler.fit_transform(df_scale),
        columns=SCALE_COLS
    )

# Plot distributions
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
scaler_names = ['Original'] + list(scalers.keys())
all_dfs = [df_scale] + list(scaled_dfs.values())

for row, col in enumerate(SCALE_COLS):
    for c_idx, (sname, sdf) in enumerate(zip(scaler_names, all_dfs)):
        ax = axes[row][c_idx]
        sdf[col].hist(ax=ax, bins=40, color=['steelblue','orange','green','red'][c_idx], alpha=0.7)
        ax.set_title(f'{col}\n{sname}', fontsize=8)
        ax.set_yticks([])

plt.suptitle('Before vs After Scaling — Distribution Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('scaling_distributions.png', dpi=150)
plt.show()

# Summary stats
print('\n=== SUMMARY STATS BY SCALER ===')
for sname, sdf in zip(scaler_names, all_dfs):
    print(f'\n--- {sname} ---')
    print(sdf[SCALE_COLS].describe().loc[['mean','std','25%','75%']].round(3))

print("""
CONCLUSION — Which Scaler is Best?
====================================
RobustScaler is best for this hotel dataset.

Reason: Columns like 'adr' and 'previous_cancellations' contain significant
outliers (luxury bookings, repeat cancellers). RobustScaler uses the median
and IQR — metrics unaffected by outliers — ensuring features are scaled without
compressing the useful variance. MinMaxScaler is vulnerable to extreme values
(one outlier can squish everything else to near-zero). StandardScaler assumes
normality which does not hold for skewed distributions like lead_time.
""")

---
## Task 4: Distance/Proximity Metrics & Impact

In [ ]:
KNN_COLS = ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights',
            'adults', 'total_of_special_requests', 'previous_cancellations', 'booking_changes']

X_knn = df[KNN_COLS].fillna(df[KNN_COLS].median())
y_knn = df[TARGET]

Xk_train, Xk_test, yk_train, yk_test = train_test_split(
    X_knn, y_knn, test_size=0.2, random_state=RANDOM_STATE, stratify=y_knn
)

results = []
configs = [
    ('No Scaling',     None,          'euclidean'),
    ('No Scaling',     None,          'manhattan'),
    ('StandardScaler', StandardScaler(), 'euclidean'),
    ('StandardScaler', StandardScaler(), 'manhattan'),
    ('RobustScaler',   RobustScaler(),   'euclidean'),
    ('RobustScaler',   RobustScaler(),   'manhattan'),
]

for scaler_name, scaler, metric in configs:
    if scaler:
        Xtr = scaler.fit_transform(Xk_train)
        Xte = scaler.transform(Xk_test)
    else:
        Xtr, Xte = Xk_train.values, Xk_test.values
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(Xtr, yk_train)
    pred = knn.predict(Xte)
    prob = knn.predict_proba(Xte)[:, 1]
    results.append({
        'Scaler': scaler_name,
        'Metric': metric,
        'Accuracy': round(accuracy_score(yk_test, pred), 4),
        'ROC-AUC': round(roc_auc_score(yk_test, prob), 4),
        'F1': round(f1_score(yk_test, pred), 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("""
OBSERVATIONS
============
1. Scaling dramatically improves KNN performance. Without scaling, 'adr' (in
   hundreds) dominates distance calculations, drowning out 'adults' or
   'booking_changes' which are single digits.

2. RobustScaler tends to outperform StandardScaler slightly because extreme
   adr values (e.g. luxury suites) are not allowed to distort the scaled space.

3. Manhattan distance performs comparably or better than Euclidean in higher
   dimensions — it is less sensitive to outliers since it does not square
   differences, making it more robust for this skewed dataset.
""")

---
## Task 5: End-to-End Numeric Pipeline

In [ ]:
from sklearn.preprocessing import FunctionTransformer

# Columns
num_features = ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights',
                'adults', 'children', 'babies', 'previous_cancellations',
                'booking_changes', 'total_of_special_requests']
cat_features = ['hotel', 'meal', 'market_segment', 'distribution_channel',
                'deposit_type', 'customer_type', 'reserved_room_type']

X_pipe = df[num_features + cat_features].copy()
y_pipe = df[TARGET]

# Log-transform pipeline for skewed columns
log_cols = ['lead_time', 'adr', 'previous_cancellations']
std_cols = [c for c in num_features if c not in log_cols]

log_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, validate=True)),
    ('scale', RobustScaler())
])

std_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', RobustScaler())
])

cat_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('log_num',  log_transformer, log_cols),
    ('std_num',  std_transformer, std_cols),
    ('cat',      cat_transformer, cat_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_pipe, y_pipe, test_size=0.2, random_state=RANDOM_STATE, stratify=y_pipe
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(full_pipeline, Xp_train, yp_train,
                             cv=cv, scoring='roc_auc')

full_pipeline.fit(Xp_train, yp_train)
yp_pred = full_pipeline.predict(Xp_test)
yp_prob = full_pipeline.predict_proba(Xp_test)[:, 1]

print('=== PIPELINE RESULTS ===')
print(f'5-Fold CV ROC-AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Test ROC-AUC      : {roc_auc_score(yp_test, yp_prob):.4f}')
print(f'Test Accuracy     : {accuracy_score(yp_test, yp_pred):.4f}')
print(f'Test F1           : {f1_score(yp_test, yp_pred):.4f}')
print('\nPipeline steps:')
print(full_pipeline)

---
## Task 6: Feature Extraction

In [ ]:
df_fe = df.copy()

# ── A) Date / Time Features ───────────────────────────────────────────────
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df_fe['arrival_month_num'] = df_fe['arrival_date_month'].map(month_map)

df_fe['arrival_season'] = pd.cut(
    df_fe['arrival_month_num'], bins=[0,3,6,9,12],
    labels=['Winter','Spring','Summer','Autumn'], include_lowest=True
)

df_fe['is_peak_season'] = df_fe['arrival_month_num'].isin([6,7,8,12]).astype(int)

df_fe['lead_time_bucket'] = pd.cut(
    df_fe['lead_time'], bins=[-1, 7, 30, 90, 180, 500],
    labels=['LastMinute','Short','Medium','Long','VeryLong']
)

df_fe['arrival_quarter'] = df_fe['arrival_month_num'].apply(
    lambda m: f'Q{(m-1)//3+1}'
)

# ── B) Pseudo-text / Categorical Encoding ─────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# Create pseudo-text feature: combine hotel type, market segment, meal
df_fe['booking_description'] = (
    df_fe['hotel'].fillna('') + ' ' +
    df_fe['market_segment'].fillna('') + ' ' +
    df_fe['meal'].fillna('')
)

tfidf = TfidfVectorizer(max_features=10)
tfidf_matrix = tfidf.fit_transform(df_fe['booking_description'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                         columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()],
                         index=df_fe.index)
df_fe = pd.concat([df_fe, tfidf_df], axis=1)

extracted_features = [
    ('arrival_month_num',  'Cyclic pattern — summer/winter peaks correlate with cancellation rates'),
    ('arrival_season',     'Seasonal demand affects booking commitment; low season → higher cancellation'),
    ('is_peak_season',     'Binary flag; peak-season bookings more likely honored due to higher prices'),
    ('lead_time_bucket',   'Categorised lead time; very long lead → more time to cancel'),
    ('arrival_quarter',    'Business vs leisure quarters influence corporate vs leisure traveller behavior'),
]
print('=== EXTRACTED FEATURES ===')
for feat, reason in extracted_features:
    print(f'  {feat:<25} → {reason}')

print(f'\nTF-IDF features added: {list(tfidf_df.columns)}')

---
## Task 7: Feature Construction

In [ ]:
df_fc = df_fe.copy()
df_fc['children'] = df_fc['children'].fillna(0)

# ── Ratio Features ────────────────────────────────────────────────────────
df_fc['price_per_person'] = df_fc['adr'] / (df_fc['adults'] + df_fc['children'] + df_fc['babies'] + 1)
df_fc['special_request_rate'] = df_fc['total_of_special_requests'] / (
    df_fc['stays_in_week_nights'] + df_fc['stays_in_weekend_nights'] + 1
)

# ── Interaction Features ──────────────────────────────────────────────────
df_fc['adr_x_lead_time']     = df_fc['adr'] * df_fc['lead_time']
df_fc['changes_x_requests']  = df_fc['booking_changes'] * df_fc['total_of_special_requests']

# ── Aggregated (Group-based) Features — computed on TRAIN ONLY ────────────
#    (leak-safe: we will recompute in the pipeline using train fold only)
df_fc['total_nights'] = df_fc['stays_in_weekend_nights'] + df_fc['stays_in_week_nights']
country_avg_adr = df_fc.groupby('country')['adr'].transform('mean')
df_fc['country_avg_adr'] = country_avg_adr

hotel_cancel_rate = df_fc.groupby('hotel')[TARGET].transform('mean')
df_fc['hotel_cancel_rate'] = hotel_cancel_rate  # leakage note applied below

# ── Polynomial / Interaction via PolynomialFeatures ───────────────────────
from sklearn.preprocessing import PolynomialFeatures
poly_input = df_fc[['lead_time', 'adr']].fillna(0)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_out = poly.fit_transform(poly_input)
poly_cols = poly.get_feature_names_out(['lead_time', 'adr'])
for i, col in enumerate(poly_cols):
    df_fc[f'poly_{col}'] = poly_out[:, i]

# ── Other useful features ─────────────────────────────────────────────────
df_fc['is_family']         = ((df_fc['children'] + df_fc['babies']) > 0).astype(int)
df_fc['is_repeat_guest']   = df_fc['is_repeated_guest']
df_fc['has_deposit']       = (df_fc['deposit_type'] != 'No Deposit').astype(int)
df_fc['prev_cancel_flag']  = (df_fc['previous_cancellations'] > 0).astype(int)

constructed = [
    'price_per_person', 'special_request_rate',
    'adr_x_lead_time', 'changes_x_requests',
    'total_nights', 'country_avg_adr',
    'poly_lead_time^2', 'poly_adr^2',
    'is_family', 'has_deposit', 'prev_cancel_flag', 'hotel_cancel_rate'
]
print('Constructed features:')
for f in constructed:
    print(f'  ✅ {f}')

print()
print('Sample:')
print(df_fc[['price_per_person','special_request_rate','adr_x_lead_time',
             'total_nights','is_family']].describe().round(2))

In [ ]:
print("""
====================================================
AVOIDING LEAKAGE IN FEATURE CONSTRUCTION
====================================================

RISK 1 — Group Aggregates Computed on Full Dataset
  Feature: country_avg_adr, hotel_cancel_rate
  Risk: If we compute the group mean over the entire dataset (train+test),
        the test rows indirectly see their own target via the group statistic.
  Prevention: Always compute group aggregates using only the training fold
        during cross-validation. In production pipelines, use
        category_encoders.TargetEncoder with cv=5 or similar to prevent
        in-fold leakage. Never use transform() across train+test together.

RISK 2 — Post-Event Features (reservation_status_date)
  Risk: This column records the date the reservation status was last changed —
        it's updated AFTER a cancellation occurs. Using it would give the model
        a direct signal of what we're trying to predict (the future).
  Prevention: Drop all features that are logically caused by or occur after
        the target event. We removed 'reservation_status' and
        'reservation_status_date' at the start.

RISK 3 — Scaling / Imputation Fitted on Full Data
  Risk: If we fit a StandardScaler or compute median imputation values using
        both train and test sets, the test set implicitly leaks information
        about its distribution into the training process.
  Prevention: Always fit scalers, imputers, and encoders ONLY on training data,
        then apply (transform) to test data. We enforce this via sklearn
        Pipeline + cross_val_score which automatically handles this correctly.
""")

---
## Task 8: Feature Importance + Selection

In [ ]:
# Build enriched feature matrix from all constructed features
FINAL_NUM_COLS = [
    'lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights',
    'adults', 'children', 'babies', 'previous_cancellations', 'booking_changes',
    'total_of_special_requests', 'price_per_person', 'special_request_rate',
    'adr_x_lead_time', 'changes_x_requests', 'total_nights',
    'poly_lead_time^2', 'poly_adr^2', 'is_family', 'has_deposit',
    'prev_cancel_flag', 'is_peak_season', 'arrival_month_num'
]

df_fs = df_fc[FINAL_NUM_COLS + CAT_COLS_BASE].copy()
df_fs[FINAL_NUM_COLS] = df_fs[FINAL_NUM_COLS].fillna(df_fs[FINAL_NUM_COLS].median())
for c in CAT_COLS_BASE:
    df_fs[c] = LabelEncoder().fit_transform(df_fs[c].astype(str))

all_feature_cols = FINAL_NUM_COLS + CAT_COLS_BASE
X_fs = df_fs[all_feature_cols]
y_fs = df[TARGET]

Xfs_train, Xfs_test, yfs_train, yfs_test = train_test_split(
    X_fs, y_fs, test_size=0.2, random_state=RANDOM_STATE, stratify=y_fs
)

In [ ]:
# ── Part A: Feature Importance ────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xfs_train, yfs_train)

rf_importances = pd.Series(rf.feature_importances_, index=all_feature_cols)
top15_rf = rf_importances.nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top15_rf.sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest Feature Importance (Top 15)')
axes[0].set_xlabel('Importance')

# Mutual Information
mi_scores = mutual_info_classif(Xfs_train, yfs_train, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=all_feature_cols)
top15_mi = mi_series.nlargest(15)

top15_mi.sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Mutual Information Score (Top 15)')
axes[1].set_xlabel('MI Score')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

overlap = set(top15_rf.index) & set(top15_mi.index)
print(f'\nOverlapping top-15 features (RF ∩ MI): {len(overlap)}')
print(sorted(overlap))

In [ ]:
# ── Part B: Filter Feature Selection ─────────────────────────────────────

# 1. Correlation filtering
corr_matrix = Xfs_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.85)]
print(f'High-correlation columns to drop (>0.85): {high_corr_cols}')

X_filtered = Xfs_train.drop(columns=high_corr_cols)
feature_cols_filtered = X_filtered.columns.tolist()

# 2. Chi-squared (need non-negative values)
X_nn = np.abs(X_filtered.values)  # ensure non-negative
chi2_scores, chi2_pvals = chi2(X_nn, yfs_train)
chi2_series = pd.Series(chi2_scores, index=feature_cols_filtered)
top15_chi2 = chi2_series.nlargest(15)
print('\nTop 15 by Chi-squared:')
print(top15_chi2.round(2))

# 3. Final feature set: union of top methods, capped at 20
final_features = list(
    (set(top15_rf.index) | set(top15_mi.index) | set(top15_chi2.index))
    - set(high_corr_cols)
)[:20]

print(f'\n=== FINAL SELECTED FEATURES ({len(final_features)}) ===')
for f in sorted(final_features):
    print(f'  {f}')

print("""
DISCUSSION
==========
RF and MI agree on lead_time, adr, deposit_type as top predictors — these
have strong non-linear relationships with cancellations (which MI captures
well). Chi-squared highlights categorical features differently (it's sensitive
to the raw scale of values). Constructed features like price_per_person and
adr_x_lead_time appear in multiple method rankings, validating their utility.
Polynomial columns (poly_lead_time^2) are removed by correlation filtering
since they're redundant with lead_time itself at low degree.
""")

---
## Final Task: Before vs After Feature Engineering Comparison

In [ ]:
def quick_eval(X_, y_, label):
    """Train LogReg with robust pipeline and return ROC-AUC and F1."""
    Xtr_, Xte_, ytr_, yte_ = train_test_split(
        X_, y_, test_size=0.2, random_state=RANDOM_STATE, stratify=y_
    )
    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  RobustScaler()),
        ('model',  LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
    ])
    pipe.fit(Xtr_, ytr_)
    prob = pipe.predict_proba(Xte_)[:, 1]
    pred = pipe.predict(Xte_)
    return roc_auc_score(yte_, prob), f1_score(yte_, pred)

# Version 1: Baseline
auc1, f1_1 = quick_eval(X.values, y.values, 'Baseline')

# Version 2: After numeric preprocessing
X2 = df_num[NUM_COLS].fillna(df_num[NUM_COLS].median())
auc2, f1_2 = quick_eval(X2.values, y.values, 'Numeric Preprocessing')

# Version 3: After extraction + construction (numeric only, from df_fc)
X3 = df_fc[FINAL_NUM_COLS].fillna(0)
auc3, f1_3 = quick_eval(X3.values, y_fs.values, 'Extraction + Construction')

# Version 4: After selection (final_features only)
available_final = [f for f in final_features if f in df_fc.columns]
X4 = df_fc[available_final].fillna(0)
auc4, f1_4 = quick_eval(X4.values, y_fs.values, 'After Selection')

comparison_data = [
    {'Version': 'Baseline',                  'Features': len(X.columns),   'Preprocessing': 'Label enc + median fill',    'Model': 'LogReg', 'ROC-AUC': round(auc1,4), 'F1': round(f1_1,4)},
    {'Version': 'Numeric Preprocessing',      'Features': len(NUM_COLS),    'Preprocessing': 'Binning + Binarize + Scale', 'Model': 'LogReg', 'ROC-AUC': round(auc2,4), 'F1': round(f1_2,4)},
    {'Version': 'Extraction + Construction',  'Features': len(FINAL_NUM_COLS),'Preprocessing': 'Log + RobustScale + Poly', 'Model': 'LogReg', 'ROC-AUC': round(auc3,4), 'F1': round(f1_3,4)},
    {'Version': 'After Selection',            'Features': len(available_final),'Preprocessing': 'Corr + Chi2 + MI filter', 'Model': 'LogReg', 'ROC-AUC': round(auc4,4), 'F1': round(f1_4,4)},
]
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
w = 0.35
ax.bar(x - w/2, comparison_df['ROC-AUC'], w, label='ROC-AUC', color='steelblue')
ax.bar(x + w/2, comparison_df['F1'],      w, label='F1',      color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Version'], rotation=15, ha='right')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Performance Across Feature Engineering Stages')
ax.legend()
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
plt.show()

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║              EXECUTIVE SUMMARY — StaySmart Hotels               ║
╠══════════════════════════════════════════════════════════════════╣

WHAT MATTERED MOST?
  The single biggest driver of performance improvement was deposit_type and
  lead_time. Guests with no deposit who book far in advance are the highest
  cancellation risk. These two features alone shift ROC-AUC significantly
  above the baseline. Feature construction (price_per_person, adr_x_lead_time)
  added compounding signal by capturing interactions invisible to linear models.

WHAT CHANGED THE PERFORMANCE CEILING?
  Moving from raw features to log-transformed + RobustScaled + constructed
  features had the most impact. The log transform of adr and lead_time
  reduced skew and brought the feature distributions closer to linearity,
  helping Logistic Regression learn better decision boundaries. Feature
  selection then removed noise without sacrificing signal, improving
  both generalisation and training stability.

MOST BUSINESS-ACTIONABLE FEATURES:
  1. deposit_type    → Non-refundable deposits drastically cut cancellations.
                       Policy lever: offer incentive to switch to deposit.
  2. lead_time       → Flag bookings >90 days out for proactive re-confirmation.
  3. prev_cancel_flag→ Repeat cancellers should face stricter deposit requirements.
  4. is_peak_season  → Peak-season bookings with no deposit warrant automated
                       reminder campaigns 2 weeks before arrival.
  5. total_nights    → Very short stays (<2 nights) are higher risk — consider
                       requiring card pre-authorisation.

RECOMMENDATION:
  Deploy a gradient-boosted model (XGBoost) with the final 20 selected features
  to score all bookings at time of confirmation. Flag top 20% risk bookings for
  retention actions (discount offers, deposit prompts, reminder calls).
╚══════════════════════════════════════════════════════════════════╝
""")